# Runtime Context（运行时上下文）

> 适用版本：本项目锁定的 **LangGraph 1.1.2**。

Runtime Context 用于向一次图运行注入用户信息、配置或依赖。它与 State 的职责不同：

- **State**：节点之间传递并更新的业务数据；
- **Context**：一次调用期间可读取的运行时依赖，不会自动写入 State。

```mermaid
flowchart LR
    I[输入 State] --> N[greet 节点]
    C[context=RuntimeContext] -. Runtime 注入 .-> N
    N --> U[返回 State 更新]
    U --> O[完整结果 State]
```


## 1. 定义 State、Context 和节点

这里使用冻结的 `dataclass` 描述 Context，表示节点只读取这份运行时数据。LangGraph 会把 `Runtime[RuntimeContext]` 自动注入节点的 `runtime` 参数。


In [1]:
from dataclasses import dataclass
from typing import TypedDict

from langgraph.graph import END, START, StateGraph
from langgraph.runtime import Runtime


class GreetingState(TypedDict):
    # State：在图中流动，并出现在最终结果中
    topic: str
    reply: str


@dataclass(frozen=True)
class RuntimeContext:
    # Context：每次 invoke 时注入的只读运行时信息
    user_name: str
    language: str


def greet(
    state: GreetingState,
    runtime: Runtime[RuntimeContext],
) -> dict[str, str]:
    context = runtime.context
    if context.language == "zh":
        reply = f"你好，{context.user_name}！你正在学习：{state['topic']}"
    else:
        reply = f"Hello, {context.user_name}! You are learning: {state['topic']}"
    return {"reply": reply}


## 2. 构建并编译图

`state_schema` 和 `context_schema` 分别声明 State 与 Runtime Context 的结构。


In [2]:
builder = StateGraph(
    state_schema=GreetingState,
    context_schema=RuntimeContext,
)
builder.add_node("greet", greet)
builder.add_edge(START, "greet")
builder.add_edge("greet", END)

graph = builder.compile()


## 3. 使用不同 Context 调用同一张图

下面保持输入 State 不变，只切换 `context=`。先完整输出两次调用返回的原始字典，再检查 Context 字段是否进入 State。


In [3]:
input_state = {"topic": "LangGraph Runtime Context"}

result_zh = graph.invoke(
    input_state,
    context=RuntimeContext(user_name="小王", language="zh"),
)
result_en = graph.invoke(
    input_state,
    context=RuntimeContext(user_name="Alice", language="en"),
)

print("中文 Context 的完整结果：")
print(result_zh)
print("\n英文 Context 的完整结果：")
print(result_en)

context_keys = {"user_name", "language"}
print("\nContext 字段是否进入 State：", bool(context_keys & result_zh.keys()))


中文 Context 的完整结果：
{'topic': 'LangGraph Runtime Context', 'reply': '你好，小王！你正在学习：LangGraph Runtime Context'}

英文 Context 的完整结果：
{'topic': 'LangGraph Runtime Context', 'reply': 'Hello, Alice! You are learning: LangGraph Runtime Context'}

Context 字段是否进入 State： False


## 结果解读

- 两次调用复用同一个 `graph`，但 `reply` 会随本次调用传入的 Context 改变；
- 返回结果只包含 `topic` 和节点写入的 `reply`，不会自动包含 `user_name`、`language`；
- 如果希望某个 Context 值成为业务状态或最终输出，节点必须显式把它写入 State；
- Runtime 还可以提供 `store`、`stream_writer` 等运行期能力，本例只聚焦最常用的 `runtime.context`。
